### Hausa  to English word pair using the modern bible as initial dataset

---
### Plan
To compare and extract sentence pairs from the translated hausa bible (Sabon Rai Don Kowa) and compare with 
modern english translated bible version and later break down to word pairs

every development would be done here

the dataset would be publicly attached here because of am still getting the hang of github pr reviews push,pull and all😅

- make sure to add 'Hausa and english datasets' via kaggle add input first

#### confirming dataset present

In [3]:
# confirming the file path exists in notebook
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/israelbajulaye/hausa-and-english-datasets/eng-web_vpl.txt
/kaggle/input/datasets/israelbajulaye/hausa-and-english-datasets/hausa_vpl.txt


In [4]:
eng_path = "/kaggle/input/datasets/israelbajulaye/hausa-and-english-datasets/eng-web_vpl.txt"
hau_path = "/kaggle/input/datasets/israelbajulaye/hausa-and-english-datasets/hausa_vpl.txt"

# Read first few lines and count total lines
with open(eng_path, 'r', encoding='utf-8') as f:
    eng_lines = f.readlines()

with open(hau_path, 'r', encoding='utf-8') as f:
    hau_lines = f.readlines()

print(f"Total English verses/lines: {len(eng_lines)}")
print(f"Total Hausa verses/lines:   {len(hau_lines)}")
print("-" * 50)

print("Sample English line 1:\n", eng_lines[0].strip())
print("\nSample Hausa line 1:\n", hau_lines[0].strip())


Total English verses/lines: 38058
Total Hausa verses/lines:   31087
--------------------------------------------------
Sample English line 1:
 GEN 1:1 In the beginning, God created the heavens and the earth.

Sample Hausa line 1:
 GEN 1:1 A farko-farko, Allah ya halicci sama da ƙasa.


#### extracting the verse key (e.g. GEN 1:1) and  finding the matching pairs, and cleaning the text:

the english bible contains Apocrypha or Septuagint not present in the hausa translated version hence the need for allignment, this removes this chapters not present in the hausa translated version

In [5]:
import re


def parse_vpl(lines):
    data = {}
    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Each line is: BOOK CHAPTER:VERSE text...
        # We split by space up to 2 times: ['GEN', '1:1', 'In the beginning...']
        parts = line.split(maxsplit=2)
        if len(parts) == 3: # confirm that the split sentence pair contains exactly 3 parts
            # seperate them into verse and scripture pair
            ref = f"{parts[0]} {parts[1]}"  # e.g., 'GEN 1:1'
            text = parts[2]
            data[ref] = text
    return data


# Parse both files
eng_dict = parse_vpl(eng_lines)
hau_dict = parse_vpl(hau_lines)

# Find matching verse IDs that exist in BOTH
common_refs = sorted(list(set(eng_dict.keys()) & set(hau_dict.keys())))

print(f"Total matching parallel verses: {len(common_refs):,}")
print("-" * 50)

# Check a sample
sample_ref = common_refs[31000]  # GEN 1:1
print(f"Sample Reference: {sample_ref}")
print(f"EN: {eng_dict[sample_ref]}")
print(f"HA: {hau_dict[sample_ref]}")

Total matching parallel verses: 31,083
--------------------------------------------------
Sample Reference: ZEC 8:19
EN: Yahweh of Armies says: “The fasts of the fourth, fifth, seventh, and tenth months shall be for the house of Judah joy, gladness, and cheerful feasts. Therefore love truth and peace.”
HA: Ga abin da Ubangiji Maɗaukaki ya ce, “Azumin wata na huɗu, da na watan biyar, da na watan bakwai da na watan goma za su zama lokutan farin ciki da murna da kuma bukukkuwa na murna ga Yahuda. Saboda haka sai ku ƙaunaci gaskiya da salama.”


In [6]:
import re
from collections import Counter, defaultdict


def clean_words(text):
    # Lowercase and retain English letters, digits, and Hausa hooked letters: ɓ, ɗ, ƙ, 'y
    text = text.lower()
    return set(re.findall(r"[a-z0-9ɓɗƙƴ']+", text))


# Step A: Build the distribution
eng_freq = Counter()
hau_freq = Counter()
co_occurrences = defaultdict(Counter)

print("Building co-occurrence distribution across all verses...")

for ref in common_refs:
    eng_words = clean_words(eng_dict[ref])
    hau_words = clean_words(hau_dict[ref])

    for e in eng_words:
        eng_freq[e] += 1
        for h in hau_words:
            co_occurrences[e][h] += 1

    for h in hau_words:
        hau_freq[h] += 1

print("Distribution complete!")


# Step B: Function to lookup the top Hausa word for any English word
def translate_word(eng_word, top_n=5, min_co_occur=3):
    eng_word = eng_word.lower()
    if eng_word not in eng_freq:
        return f"'{eng_word}' not found in English corpus."

    candidates = co_occurrences[eng_word]
    scored = []

    for hau_word, joint_count in candidates.items():
        if joint_count < min_co_occur:
            continue
        # Dice score penalizes stopwords like 'da', 'ya', 'a', 'ne'
        dice_score = (2.0 * joint_count) / (eng_freq[eng_word] + hau_freq[hau_word])
        scored.append((hau_word, dice_score, joint_count))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]

Building co-occurrence distribution across all verses...
Distribution complete!


 #### the implementation of  co-occurrence distribution logic (with the Dice coefficient penalty for stopwords).
 basically every single word(the english dataset) is cross referenced with every other word in each scripture in the hausa dataset
 then a frequency distribution of each word relative to the other words is formed
 stop word are then penalised via the dice coefficient penalty
 this should produce a accurate word to word translation

In [7]:
test_words = ["god", "earth", "heavens", "covenant", "water", "king", "faith"]

for word in test_words:
    results = translate_word(word, top_n=3)
    print(f"\nEnglish: '{word}' (Total EN verses: {eng_freq[word]})")
    for hau_word, score, count in results:
        print(
            f"   → Hausa: '{hau_word}' | Score: {score:.3f} | Co-occurred: {count} times"
        )



English: 'god' (Total EN verses: 3574)
   → Hausa: 'allah' | Score: 0.808 | Co-occurred: 2621 times
   → Hausa: 'ubangiji' | Score: 0.250 | Co-occurred: 1254 times
   → Hausa: 'ya' | Score: 0.225 | Co-occurred: 2123 times

English: 'earth' (Total EN verses: 870)
   → Hausa: 'duniya' | Score: 0.668 | Co-occurred: 565 times
   → Hausa: 'ƙasa' | Score: 0.287 | Co-occurred: 244 times
   → Hausa: 'sama' | Score: 0.192 | Co-occurred: 167 times

English: 'heavens' (Total EN verses: 145)
   → Hausa: 'sammai' | Score: 0.657 | Co-occurred: 91 times
   → Hausa: 'sararin' | Score: 0.137 | Co-occurred: 17 times
   → Hausa: 'duniya' | Score: 0.128 | Co-occurred: 62 times

English: 'covenant' (Total EN verses: 322)
   → Hausa: 'alkawari' | Score: 0.405 | Co-occurred: 130 times
   → Hausa: 'alkawarin' | Score: 0.351 | Co-occurred: 100 times
   → Hausa: 'akwatin' | Score: 0.255 | Co-occurred: 65 times

English: 'water' (Total EN verses: 373)
   → Hausa: 'ruwa' | Score: 0.558 | Co-occurred: 242 times
 

#### to make sure this is even more accurate and also to stay true to word to word pair, am adding a bidirectional confirmation

each english word is translated to hausa , and also verified that it's hausa word translation to english is the same thing bidirectionally

In [8]:
# Function 1: Hausa -> English
def translate_hau_to_eng(hau_word, top_n=5, min_co_occur=3):
    hau_word = hau_word.lower()
    if hau_word not in hau_freq:
        return f"'{hau_word}' not found in Hausa corpus."

    # Invert the search: check all English words that co-occurred with hau_word
    scored = []
    for eng_word, candidates in co_occurrences.items():
        if hau_word in candidates:
            joint_count = candidates[hau_word]
            if joint_count < min_co_occur:
                continue

            dice_score = (2.0 * joint_count) / (
                eng_freq[eng_word] + hau_freq[hau_word]
            )
            scored.append((eng_word, dice_score, joint_count))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]


# Function 2: Mutual Agreement (Bidirectional Verifier)
def verify_bidirectional_match(eng_word, top_k=3):
    """Checks if English -> Hausa candidate also maps back to English in the top-k."""
    eng_word = eng_word.lower()
    forward_matches = translate_word(eng_word, top_n=top_k)

    if isinstance(forward_matches, str):
        return forward_matches

    verified_results = []

    for hau_cand, fwd_score, fwd_count in forward_matches:
        # Check backward direction
        backward_matches = translate_hau_to_eng(hau_cand, top_n=top_k)

        is_mutual = False
        back_rank = None

        if isinstance(backward_matches, list):
            for rank, (back_eng, back_score, _) in enumerate(
                backward_matches, start=1
            ):
                if back_eng == eng_word:
                    is_mutual = True
                    back_rank = rank
                    break

        verified_results.append(
            {
                "hausa": hau_cand,
                "score": fwd_score,
                "co_count": fwd_count,
                "mutual_verified": is_mutual,
                "backward_rank": back_rank,
            }
        )

    return verified_results

In [9]:
hausa_test_words = ["mutum", "haske", "hanya", "gaskiya", "ruwa"]

for word in hausa_test_words:
    results = translate_hau_to_eng(word, top_n=3)
    print(f"\nHausa: '{word}' (Total HA verses: {hau_freq[word]})")
    for eng_word, score, count in results:
        print(
            f"   → English: '{eng_word}' | Score: {score:.3f} | Co-occurred: {count} times"
        )


Hausa: 'mutum' (Total HA verses: 1347)
   → English: 'man' | Score: 0.509 | Co-occurred: 864 times
   → English: 'a' | Score: 0.150 | Co-occurred: 583 times
   → English: 'son' | Score: 0.150 | Co-occurred: 235 times

Hausa: 'haske' (Total HA verses: 138)
   → English: 'light' | Score: 0.583 | Co-occurred: 104 times
   → English: 'darkness' | Score: 0.310 | Co-occurred: 44 times
   → English: 'lamp' | Score: 0.131 | Co-occurred: 14 times

Hausa: 'hanya' (Total HA verses: 259)
   → English: 'way' | Score: 0.259 | Co-occurred: 128 times
   → English: 'road' | Score: 0.136 | Co-occurred: 19 times
   → English: 'path' | Score: 0.113 | Co-occurred: 17 times

Hausa: 'gaskiya' (Total HA verses: 483)
   → English: 'truth' | Score: 0.351 | Co-occurred: 120 times
   → English: 'certainly' | Score: 0.242 | Co-occurred: 74 times
   → English: 'most' | Score: 0.209 | Co-occurred: 75 times

Hausa: 'ruwa' (Total HA verses: 494)
   → English: 'water' | Score: 0.558 | Co-occurred: 242 times
   → Engli

In [10]:
test_eng_words = [
    "covenant",
    "light",
    "peace",
    "truth",
    "spirit",
    "heart",
    "father",
]

for word in test_eng_words:
    results = verify_bidirectional_match(word, top_k=3)
    print(f"\nEnglish Word: '{word}'")
    for res in results:
        status = (
            f"✅ VERIFIED (maps back at rank #{res['backward_rank']})"
            if res["mutual_verified"]
            else "❌ ONE-WAY ONLY"
        )
        print(
            f"   → Hausa: '{res['hausa']:<15}' | Score: {res['score']:.3f} | {status}"
        )


English Word: 'covenant'
   → Hausa: 'alkawari       ' | Score: 0.405 | ✅ VERIFIED (maps back at rank #1)
   → Hausa: 'alkawarin      ' | Score: 0.351 | ✅ VERIFIED (maps back at rank #2)
   → Hausa: 'akwatin        ' | Score: 0.255 | ✅ VERIFIED (maps back at rank #2)

English Word: 'light'
   → Hausa: 'haske          ' | Score: 0.583 | ✅ VERIFIED (maps back at rank #1)
   → Hausa: 'hasken         ' | Score: 0.336 | ✅ VERIFIED (maps back at rank #1)
   → Hausa: 'duhu           ' | Score: 0.303 | ✅ VERIFIED (maps back at rank #2)

English Word: 'peace'
   → Hausa: 'salama         ' | Score: 0.731 | ✅ VERIFIED (maps back at rank #1)
   → Hausa: 'lafiya         ' | Score: 0.215 | ✅ VERIFIED (maps back at rank #2)
   → Hausa: 'hadaya         ' | Score: 0.130 | ❌ ONE-WAY ONLY

English Word: 'truth'
   → Hausa: 'gaskiya        ' | Score: 0.351 | ✅ VERIFIED (maps back at rank #1)
   → Hausa: 'aminci         ' | Score: 0.117 | ❌ ONE-WAY ONLY
   → Hausa: 'gaskiyar       ' | Score: 0.116 | ✅ VER

## to allign with lalango specification

In [11]:
import os
import random
import shutil
import zipfile
from collections import Counter, defaultdict

# -------------------------------------------------------------
# 1. PREPARE RAW DATA (Scripture sentences without verse IDs)
# -------------------------------------------------------------
raw_dir = "/kaggle/working/data/raw/hausa-english"
os.makedirs(raw_dir, exist_ok=True)

raw_pairs = []
for ref in common_refs:
    en_verse = eng_dict[ref].strip()
    ha_verse = hau_dict[ref].strip()
    if en_verse and ha_verse:
        # Source = Hausa, Target = English
        raw_pairs.append((ha_verse, en_verse))

# Shuffle reproducible
random.seed(42)
random.shuffle(raw_pairs)

total_raw = len(raw_pairs)
train_end = int(total_raw * 0.8)
val_end = int(total_raw * 0.9)

raw_train = raw_pairs[:train_end]
raw_val = raw_pairs[train_end:val_end]
raw_test = raw_pairs[val_end:]


def write_parallel_files(pairs, folder, split_name):
    src_file = os.path.join(folder, f"{split_name}.src")
    tgt_file = os.path.join(folder, f"{split_name}.tgt")
    with open(src_file, "w", encoding="utf-8") as f_src, open(
        tgt_file, "w", encoding="utf-8"
    ) as f_tgt:
        for src, tgt in pairs:
            f_src.write(src.strip() + "\n")
            f_tgt.write(tgt.strip() + "\n")


# Write raw files
write_parallel_files(raw_pairs, raw_dir, "all")
write_parallel_files(raw_train, raw_dir, "train")
write_parallel_files(raw_val, raw_dir, "val")
write_parallel_files(raw_test, raw_dir, "test")

print(f"✅ Raw scripture pairs created: {total_raw:,}")
print(f"   - Train: {len(raw_train):,} | Val: {len(raw_val):,} | Test: {len(raw_test):,}")

# -------------------------------------------------------------
# 2. PREPARE PROCESSED DATA (Bidirectional Word-to-Word Matching)
# -------------------------------------------------------------
processed_dir = "/kaggle/working/data/processed/hausa-english"
os.makedirs(processed_dir, exist_ok=True)

print("\nExtracting mutually verified word-to-word pairs...")

# Step A: Best Hausa for each English word (Forward)
best_hau_for_eng = {}
for e, h_counts in co_occurrences.items():
    if eng_freq[e] < 3:
        continue
    best_h = None
    best_score = -1.0
    for h, joint in h_counts.items():
        if joint < 3:
            continue
        dice = (2.0 * joint) / (eng_freq[e] + hau_freq[h])
        if dice > best_score:
            best_score = dice
            best_h = h
    if best_h:
        best_hau_for_eng[e] = (best_h, best_score)

# Step B: Best English for each Hausa word (Backward)
# Invert co_occurrences
hau_to_eng_candidates = defaultdict(Counter)
for e, h_counts in co_occurrences.items():
    for h, count in h_counts.items():
        hau_to_eng_candidates[h][e] = count

best_eng_for_hau = {}
for h, e_counts in hau_to_eng_candidates.items():
    if hau_freq[h] < 3:
        continue
    best_e = None
    best_score = -1.0
    for e, joint in e_counts.items():
        if joint < 3:
            continue
        dice = (2.0 * joint) / (eng_freq[e] + hau_freq[h])
        if dice > best_score:
            best_score = dice
            best_e = e
    if best_e:
        best_eng_for_hau[h] = (best_e, best_score)

# Step C: Mutual agreement intersection (E -> H and H -> E must agree)
verified_word_pairs = []
for eng_word, (hau_match, score) in best_hau_for_eng.items():
    if hau_match in best_eng_for_hau:
        reverse_eng, rev_score = best_eng_for_hau[hau_match]
        if reverse_eng == eng_word:
            # Source: Hausa word, Target: English word
            verified_word_pairs.append((hau_match, eng_word, score))

# Sort by confidence score descending
verified_word_pairs.sort(key=lambda x: x[2], reverse=True)

# Prepare pure word pairs (src, tgt)
processed_pairs = [(h, e) for h, e, _ in verified_word_pairs]

# Split processed word pairs
random.shuffle(processed_pairs)
total_proc = len(processed_pairs)
proc_train_end = int(total_proc * 0.8)
proc_val_end = int(total_proc * 0.9)

proc_train = processed_pairs[:proc_train_end]
proc_val = processed_pairs[proc_train_end:proc_val_end]
proc_test = processed_pairs[proc_val_end:]

write_parallel_files(processed_pairs, processed_dir, "all")
write_parallel_files(proc_train, processed_dir, "train")
write_parallel_files(proc_val, processed_dir, "val")
write_parallel_files(proc_test, processed_dir, "test")

print(f"✅ Mutually verified word pairs created: {total_proc:,}")
print(
    f"   - Train: {len(proc_train):,} | Val: {len(proc_val):,} | Test: {len(proc_test):,}"
)


# -------------------------------------------------------------
# 3. ZIP EVERYTHING INTO THE EXACT FOLDER STRUCTURE
# -------------------------------------------------------------
zip_output_path = "/kaggle/working/data_hausa_english.zip"
with zipfile.ZipFile(zip_output_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    base_dir = "/kaggle/working/data"
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            full_path = os.path.join(root, file)
            # Relative path preserves data/raw/... and data/processed/...
            rel_path = os.path.relpath(full_path, "/kaggle/working")
            zipf.write(full_path, rel_path)

print(f"\n📦 Done! Complete data package zipped at: {zip_output_path}")

✅ Raw scripture pairs created: 31,083
   - Train: 24,866 | Val: 3,108 | Test: 3,109

Extracting mutually verified word-to-word pairs...
✅ Mutually verified word pairs created: 3,062
   - Train: 2,449 | Val: 306 | Test: 307

📦 Done! Complete data package zipped at: /kaggle/working/data_hausa_english.zip


to expand from 3000 words to about 8000

we can 

Allow Many-to-One: Let multiple English words map to the same Hausa word (e.g., earth, land, and ground can all keep ƙasa).
Top-3 Mutual Agreement: Instead of requiring Rank #1 both ways, accept if they are within each other's Top 3.
Lower threshold to 2 occurrences: Include words that appear at least twice.

In [16]:
import re

# 1. Build bidirectional lookup maps from your 3,062 verified word pairs
# In verified_word_pairs, index 0 is Hausa, index 1 is English
eng_to_hau = {eng.lower(): hau for hau, eng, _ in verified_word_pairs}
hau_to_eng = {hau.lower(): eng for hau, eng, _ in verified_word_pairs}

print(f"Loaded {len(eng_to_hau):,} bidirectional dictionary pairs.")


# 2. Translation engine with OOV (Out-of-Vocabulary) fallback
def translate_sentence(sentence, direction="en->ha"):
    lookup = eng_to_hau if direction == "en->ha" else hau_to_eng

    matched_words = []
    unmatched_words = []

    def replace_word(match):
        token = match.group(0)
        lower_token = token.lower()

        if lower_token in lookup:
            translated = lookup[lower_token]
            matched_words.append(token)

            # Preserve original casing
            if token.isupper():
                return translated.upper()
            elif token.istitle():
                return translated.capitalize()
            else:
                return translated
        else:
            # Fallback: keep original word
            unmatched_words.append(token)
            return token

    # Matches words including Hausa hooked letters (ɓ, ɗ, ƙ, 'y)
    translated_text = re.sub(r"[a-zA-Z0-9ɓɗƙƴ']+", replace_word, sentence)

    total_words = len(matched_words) + len(unmatched_words)
    coverage = (
        (len(matched_words) / total_words * 100) if total_words > 0 else 0
    )

    return translated_text, coverage, unmatched_words


# 3. Interactive CLI Prompter
def run_interactive_translator():
    print("=" * 65)
    print("🌍 La Lango Word-by-Word Translator (3,000+ Verified Lexicon)")
    print("Type 'q' or 'exit' anytime to stop.")
    print("=" * 65)

    while True:
        choice = (
            input(
                "\nSelect Direction:\n  [1] English ➔ Hausa\n  [2] Hausa ➔ English\nChoice (1 or 2): "
            )
            .strip()
            .lower()
        )

        if choice in ["q", "exit"]:
            print("Goodbye!")
            break

        if choice in ["1", "en", "english"]:
            direction = "en->ha"
            src_lang, tgt_lang = "English", "Hausa"
        elif choice in ["2", "ha", "hausa"]:
            direction = "ha->en"
            src_lang, tgt_lang = "Hausa", "English"
        else:
            print("⚠️ Invalid choice. Please enter '1' or '2'.")
            continue

        sentence = input(f"\nEnter {src_lang} sentence: ").strip()
        if sentence.lower() in ["q", "exit"]:
            print("Goodbye!")
            break

        if not sentence:
            continue

        translation, coverage, oov = translate_sentence(sentence, direction)

        print("-" * 50)
        print(f"📖 Original ({src_lang}):    {sentence}")
        print(f"✨ Translation ({tgt_lang}): {translation}")
        print(f"📊 Dictionary Coverage: {coverage:.1f}%")
        if oov:
            print(f"ℹ️ Words kept in original language (not in dict): {oov}")
        print("-" * 50)

Loaded 3,062 bidirectional dictionary pairs.


In [ ]:
run_interactive_translator()

In [17]:
import os
import zipfile
import pandas as pd

# Create output folder for CSVs
csv_dir = "/kaggle/working/data_csv"
os.makedirs(csv_dir, exist_ok=True)

# -------------------------------------------------------------
# 1. SAVE WORD-TO-WORD VERIFIED DICTIONARY (3,062 Words)
# -------------------------------------------------------------
dict_data = [
    {"hausa": h, "english": e, "confidence_score": round(score, 4)}
    for h, e, score in verified_word_pairs
]

df_dict = pd.DataFrame(dict_data)
dict_csv_path = os.path.join(csv_dir, "hausa_english_dictionary.csv")
df_dict.to_csv(dict_csv_path, index=False, encoding="utf-8")
print(
    f"✅ Saved dictionary CSV: {dict_csv_path} ({len(df_dict):,} word pairs)"
)

# -------------------------------------------------------------
# 2. SAVE PARALLEL SENTENCE SPLITS (31,083 Verses)
# -------------------------------------------------------------
df_train = pd.DataFrame(raw_train, columns=["hausa", "english"])
df_val = pd.DataFrame(raw_val, columns=["hausa", "english"])
df_test = pd.DataFrame(raw_test, columns=["hausa", "english"])
df_all = pd.DataFrame(raw_pairs, columns=["hausa", "english"])

train_csv_path = os.path.join(csv_dir, "train.csv")
val_csv_path = os.path.join(csv_dir, "val.csv")
test_csv_path = os.path.join(csv_dir, "test.csv")
all_csv_path = os.path.join(csv_dir, "all_sentences.csv")

df_train.to_csv(train_csv_path, index=False, encoding="utf-8")
df_val.to_csv(val_csv_path, index=False, encoding="utf-8")
df_test.to_csv(test_csv_path, index=False, encoding="utf-8")
df_all.to_csv(all_csv_path, index=False, encoding="utf-8")

print(f"✅ Saved train.csv: {len(df_train):,} rows")
print(f"✅ Saved val.csv:   {len(df_val):,} rows")
print(f"✅ Saved test.csv:  {len(df_test):,} rows")
print(f"✅ Saved all_sentences.csv: {len(df_all):,} rows")

# -------------------------------------------------------------
# 3. ZIP EVERYTHING FOR 1-CLICK DOWNLOAD
# -------------------------------------------------------------
zip_csv_path = "/kaggle/working/hausa_english_csv_package.zip"
with zipfile.ZipFile(zip_csv_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in os.listdir(csv_dir):
        file_path = os.path.join(csv_dir, filename)
        zipf.write(file_path, arcname=filename)

print(f"\n📦 All CSVs zipped and ready for download at: {zip_csv_path}")

✅ Saved dictionary CSV: /kaggle/working/data_csv/hausa_english_dictionary.csv (3,062 word pairs)
✅ Saved train.csv: 24,866 rows
✅ Saved val.csv:   3,108 rows
✅ Saved test.csv:  3,109 rows
✅ Saved all_sentences.csv: 31,083 rows

📦 All CSVs zipped and ready for download at: /kaggle/working/hausa_english_csv_package.zip
